Introdução

Representação de grafos

In [ ]:
from collections import deque

# Class Vertice
class Vertex:
    ''' Estrutura de Vértice para um grafo: encapsula um elemento (vertex_id) 
        que é o identificador deste nó.
        
        O elemento (vertex_id) deve ser hashable:  
        - Um objeto hashable é aquele que pode ser utilizado como uma chave num dicionário Python.
        - Isto inclui strings, números, tuplas, etc.
    '''
    
    def __init__(self, vertex_id):
        '''O vértice será inserido no Grafo usando o método insert_vertex(x) que cria um Vertex'''
        self._vertex_id = vertex_id   # Id do vértice (elemento a inserir no grafo)
        
    def __hash__(self):
        '''O valor do elemento é usado como hash para o vértice (o elemento deve ser hashable)'''
        return hash(self._vertex_id)  # devolve o hash do elemento

    def __str__(self):
        '''Devolve a representação do objeto vértice em string.'''
        return'v{0}'.format(self._vertex_id)

    def __eq__(self, vertex):
        return vertex_id == vertex._vertex_id # Deve-se garantir que: se hash(vertex)==hash(self), entao vertex==self

    def __lt__(self, vertex):
        return vertex_id < vertex._vertex_id
    
    def __le__(self, vertex):
        return self._vertex_id <= vertex._vertex_id
    
    def __gt__(self, vertex):
        return self._vertex_id > vertex._vertex_id
    
    def __ge__(self, vertex):
        return self._vertex_id >= vertex._vertex_id
    
    def vertex_id(self):
        ''' Devolve o elemento guardado neste vértice.'''
        return self._vertex_id

In [ ]:
# Class Edge
class Edge:
    ''' Estrutura de Aresta para um Grafo: (origem, destino) e peso '''

    def __init__(self, vertex_1, vertex_2, weight):
        self._vertex_1 = vertex_1
        self._vertex_2 = vertex_2
        self._weight = weight

    def __hash__(self):
        # Função que mapeia a aresta a uma posição no dicionário (hash map)
        return hash( (self._vertex_1, self._vertex_2) )

    def __str__(self):
        ''' Devolve a representação do objeto aresta em string: (origem, destino)w=peso '''
        return'e({0},{1})w={2}'.format(self._vertex_1, self._vertex_2, self._weight)

    def __eq__(self, other):
        # define igualdade de duas arestas (deve ser consistente com a função hash)
        return self._vertex_1 == other._vertex_1 and self._vertex_2 == other._vertex_2

    def endpoints(self):
        ''' Devolve a tupla (vertex_1, vertex_2) os vértices adjacentes vertex_1 e vertex_2.'''
        return (self._vertex_1, self._vertex_2)

    def cost(self):
        ''' Devolve o peso associado a este arco.'''
        return self._weight
    
    def opposite(self, vertex):
        ''' Indica o vértice oposto ao vertex nesta aresta 
            (apenas se vertex fizer parte da aresta).'''
        if vertex == self._vertex_1:
            return self._vertex_2
        elif vertex == self._vertex_2:
            return self._vertex_1
        else:
            return None

In [ ]:
class Graph:
    '''
    Representação de um grafo usando dicionários encadeados (nested dictionaries).
    
    Atributos:
    ----------
    adjancencies: Dicionário externo que associa um vértice (Vertex) a um  
                  mapa de adjacências (dicionario interno)
                  Em vez de utilizar uma estrutura com um array (como nos slides),
                  utiliza uma estrutura do tipo "dicionário de dicionários":
                  < vertex , < neighbor_vertex , edge > >
                  
    vertices: Dicionário auxiliar que associa o id dos vértices do grafo
              a um objeto Vertex (tabela de símbolos):
              < vertex_id , vertex >
              
    n: Número de vértices no Grafo
    m: Número de arestas no Grafo
    
    ----------
'''
    def __init__(self):
        '''Construtor: Cria um grafo vazio (dicionário de _adjancencies).'''
        self._adjancencies = {}  # dicionário que associa o par chave-valor: <Vertex v, Mapa de adjacências de v> 
        self._vertices = {}      # dicionário que associa o par: <id do vértice, objeto Vertex correspondente> 
        self._n = 0              # número de vértices do grafo
        self._m = 0              # número de arestas do grafo
        
    def __str__(self):
        '''Devolve a representação do grafo em string (toString)'''
        if self._n == 0:
            ret = "DAA-Graph: <empty>\n"
        else:
            ret = "DAA-Graph:\n"
            for vertex in self._adjancencies.keys():
                #ret += "vertex-"
                ret += str(vertex) + ": "
                for edge in self.incident_edges(vertex.vertex_id()):
                    ret += str(edge) + "; "
                ret += "\n"
        return ret
    
    def is_directed(self):
        '''A classe Graph representa um grafo não orientado.'''
        return False
    
    def order(self):
        '''Ordem de um grafo: a quantidade de vértices no Grafo.'''
        return self._n
    
    def size(self):
        '''Dimensão de um grafo: a quantidade total de arestas do Grafo.'''
        return self._m 
    
    def has_vertex(self, vertex_id):
        '''Verifica se o vértice de id vertex_id está no grafo.'''
        return vertex_id in self._vertices
    
    def has_edge(self, u_id, v_id):
        '''Verifica se a aresta (u_id, v_id) existe no grafo.'''
        if not self.has_vertex(u_id) or not self.has_vertex(v_id):
            return False
        else:
            vertex_u = self._vertices[u_id]
            vertex_v = self._vertices[v_id]
            return vertex_v in self._adjancencies[vertex_u]
        
    def insert_vertex(self, vertex_id):
        '''Insere um novo vértice com o id vertex_id.'''
        if not self.has_vertex(vertex_id):
            vertex = Vertex(vertex_id)          # instancia um objeto do tipo Vertex
            self._vertices[vertex_id] = vertex  # insere o novo vértice no dicionário de vertices
            self._adjancencies[vertex] = {}     # inicializa o mapa de adjacências deste vértice a vazio
            self._n +=1                         # mais um vértice no grafo

    def insert_edge(self, u_id, v_id, weight=0):
        ''' Cria e insere uma nova aresta entre u_id e v_id com peso weight.
            Se a aresta já existe no grafo, atualiza-se o seu peso.
            Também insere os vértices u_id e v_id, caso não existam.'''
        if not self.has_vertex(u_id):
            self.insert_vertex(u_id) # insere novo vertex e atualiza n
        if not self.has_vertex(v_id):
            self.insert_vertex(v_id) # insere novo vertex e atualiza n      
        if not self.has_edge(u_id, v_id):
            self._m +=1           # atualiza m apenas se a aresta ainda não existir no grafo
        else:
            print(f"Existing edge {u_id} and {v_id}. Will only update weight")
        vertex_u = self._vertices[u_id]  # acede ao objeto Vertex associado a u_id
        vertex_v = self._vertices[v_id]  # acede ao objeto Vertex associado a v_id
        e = Edge(vertex_u, vertex_v, weight)    
        self._adjancencies[vertex_u][vertex_v] = e  # coloca v nas adjacências de u
        self._adjancencies[vertex_v][vertex_u] = e  # e u nas adjacências de v (para facilitar a procura de todas as arestas incidentes num vértice)
    
    def degree(self, vertex_id):
        '''Quantidade de arestas incidentes no vértice v.
        '''
        return len(self._adjancencies[self._vertices[vertex_id]])
    
    def get_vertex(self, vertex_id):
        ''' Devolve o objeto Vertex associado ao elemento vertex_id no grafo
        '''
        return None if not self.has_vertex(vertex_id) else self._vertices[vertex_id] 
    
    def get_edge(self, u_id, v_id):
        ''' Devolve o objeto aresta (Edge) que liga u_id a v_id. 
            Devolve None se não forem adjacentes ou se (um d)os vértices não existirem.'''  
        if not self.has_edge(u_id, v_id):
            return None
        else:
            vertex_u = self._vertices[u_id]
            vertex_v = self._vertices[v_id]
            return self._adjancencies[vertex_u][vertex_v]
    
    def vertices(self):
        '''Devolve um iterável sobre todos os vértices do Grafo (tipo Vertex)'''
        return self._vertices.values()

    def edges(self):
        '''Devolve um iterável sobre todas as arestas do Grafo (sem arestas duplicadas).'''
        seen = {}      # evita a repetição de arestas no grafo não orientado
        for adj_map in self._adjancencies.values():
            for edge in adj_map.values():
                if edge not in seen:
                    yield edge
                seen[edge] = True

    def incident_edges(self, vertex_id):
        '''Devolve um iterável (gerador) com todas as arestas de um vértice com id vertex_id.'''
        vertex = self._vertices[vertex_id]
        for edge in self._adjancencies[vertex].values(): # para todas as arestas incidentes em v:
            yield edge

    def has_neighbors(self, vertex_id):
        '''Verifica se o vértice de id vertex_id tem vértices adjacentes (vizinhos).'''
        if not self.has_vertex(vertex_id):
            return False
        return self.degree(vertex_id) > 0
    
    def remove_vertex(self, vertex_id):
        '''Remove o vértice com id vertex_id. Se o vértice não existir, não faz nada.'''
        # Passo 1: remover todas as arestas do vértice dado
        # Passo 2: remover todas as arestas incidentes em vertex_id dos mapas de outros vertices
        # Passo 3: remover o vértice com id vertex_id do grafo
        # Passo 4: decrementa contador de vértices
        if self.has_vertex(vertex_id):
            lst_copied = list(self.incident_edges(vertex_id)) # copia para a lista para evitar erros de concorrência (remove enquanto itera na lista)
            for edge in lst_copied:
                x, y = edge.endpoints()
                self.remove_edge(x.vertex_id(),y.vertex_id())  # (Passos 1 e 2)
            del self._adjancencies[self._vertices[vertex_id]]  # (Passo 3 - remove do dicionário de adjacências)
            del self._vertices[vertex_id]                      # (Passo 3 - remove do dicionário de vértices)
            self._n -=1                                        # (Passo 4 - decrementa contador)
        
    def remove_edge(self, u_id, v_id):
        '''Remove a aresta entre u_id e v_id. Se a aresta não existir, não faz nada.'''
        if  self.has_edge(u_id, v_id):
            vertex_u = self._vertices[u_id]
            vertex_v = self._vertices[v_id]
            del self._adjancencies[vertex_u][vertex_v]
            if vertex_u != vertex_v:  # laços são removidos apenas uma vez
                del self._adjancencies[vertex_v][vertex_u]
            self._m -=1


API ConnectedComponents

In [ ]:
class ConnectedComponents:
    
    def __init__(self, graph):
        marked = [False] * graph.order()
        self._graph = graph
        self._id = [None] * graph.order()
        
        self._count = 0
        for v in self._graph.vertices():
            if not self._marked[v]:
                vid = v.vertex_id()
                self._id[vid] = self._count
                self.dfs(vid)
            self._count += 1

    def dfs(self, source):
        
        stack = deque()
        stack.append(source)

        while stack:
            self._id[v] = self._id[source]
            v = stack.pop()
            self._marked[v] = True
            for edge in self._graph.incident_edges(v):
                v2 = edge.opposite(v)
                if not self._marked[v2]:
                    stack.append(v2)
        
    def connected(self, u, v):
        return self._id[u] == self._id[v]
    
    def component_id(self, v):
        return self._id[v]
    
    def count(self, v):
        return self._count
    
    def largest_component(self):
        
        largest_id = None
        count_comps = 0
        
        
        for i in range(self.count()):
            count = 0
            for j in range(self._graph.order()):
                if self._id[j] == i:
                    count += 1
            if count > count_comps:
                count_comps = count
                largest_id = i
        
        graph = Graph()
        for i in range(self._graph.order()):
            if self._id[i] == largest_id:
                graph.insert_vertex(i)

        for i in graph.indices():
            for j in self._graph.indices():
                if self._graph.has_edge(i, j):
                    graph.insert_edge(i, j)

        return graph



SyntaxError: incomplete input (2038450445.py, line 33)

API CentralityAnalyzer

In [ ]:
class CentralityAnalyzer:
    def __init__(self, graph: Graph):
        self._graph = graph
        self._num_comps = 0
        self._con_comps = None
        self._largest_comp = None
    
    def bfs(self, source):
        
        # Inicializacao de estruturas auxiliares
        marked = {}
        parents = {}
        distTo = {}
        sigma = {}

        for i in range(self._graph.order):
            marked[i] = False
            parents[i] = list()
            distTo[i] = -1
            sigma[i] = 0

        #Inicializacao da Pilha para a betweeness_centrality
        stack = deque()
        
        #Inicializacao da Queue
        queue = deque()

        #Inicio com o vertice source
        marked[source] = True
        queue.append(source)
        distTo[source] = 0
        sigma[source] = 1

        while queue:
            v = queue.popleft()
            stack.append(v)
            for edge in self._graph.incident_edges(v):
                v2 = edge.opposite(v)
                #Primeiro caminho encontrado
                if not marked[v2]:
                    marked[v2] = True
                    queue.append(v2)
                    parents[v2].append(v)
                    distTo[v2] = distTo[v] + 1
                    sigma[v2] = sigma[v]
                
                #Outro caminho minimo encontrado
                elif distTo[v2] == distTo[v] + 1:
                    sigma[v2] += sigma[v]
                    parents[v2].append(v)

        return parents, distTo, sigma, stack
    
    def num_components(self):
        if self._con_comps is None:
            self._con_comps = ConnectedComponents(self._graph)
        return self._con_comps.count()
    
    def largest_component(self):
        if self._largest_comp is None:
            if self._con_comps is None:
                self._con_comps = ConnectedComponents(self._graph)
            self._largest_comp = self._con_comps.largest_component()
        return self._con_comps.largest_component()

    def degree_distribution(self):
        degree_dic = {}
        for v in self._graph.vertices:
            val = self._graph.degree(v.vertex_id())
            if val not in degree_dic:
                degree_dic[val] = 1
            else:
                degree_dic[val] += 1
        return degree_dic
    
    def diameter(self):
        graph = self.largest_component()
        largest_dist = 0
        for v in graph.vertices():
            (_, dist) = self.bfs(v.vertex_id())
            
            #Excentricidade: max d(v, v2)
            excentricity = 0
            for v2 in graph.vertices():
                val = dist[v2.vertex_id()]
                if val > excentricity:
                    excentricity = val
            if(excentricity > largest_dist):
                largest_dist = excentricity
        return largest_dist
    
    def degree_centrality(self):
        dc_dic = {}
        for v in self._graph.vertices():
            id = v.vertex_id()
            dc_dic[id] = self._graph.degree(id) / (self._graph.order() - 1)
        return dc_dic
    
    def closeness_centrality(self):
        cc_dic = {}
        for v in self._graph.vertices():
            (_, dist) = self.bfs(v.vertex_id())
            rv = 0
            sum = 0
            for d in dist.values():
                if d > 0:
                    rv += 1
                    sum += d
            if rv == 0:
                cc_dic[v.vertex_id()] = 0
            else:
                cc_dic[v.vertex_id()] = rv**2 / ((self._graph.order() - 1) * sum)
        return cc_dic
    
    def eigenvector_centrality(self):
        scores = {v.vertex_id(): 1 for v in self._graph.vertices()}
        new_scores = {}
        epsilon = 0.000001
        if self._largest_comp is None:
            if self._con_comps is None:
                self._con_comps = ConnectedComponents(self._graph)
            self._largest_comp = self._con_comps.largest_component()
        lc = self._largest_comp
        for v in self._graph.vertices():
            if not lc.has_vertex(v.vertex_id()):
                scores[v.vertex_id()] = 0
        for it in range(100):
            new_scores = {}
            for v in lc.vertices():
                sum_v = 0
                for edge in self._graph.incident_edges(v.vertex_id()):
                    v2 = edge.opposite(v.vertex_id())
                    sum_v += scores[v2.vertex_id()]
                new_scores[v.vertex_id()] = sum_v
            max_s = max(new_scores.values())
            
            # So para seguranca
            if max_s == 0:
                return new_scores, it + 1
            
            for v in lc.vertices():
                new_scores[v.vertex_id()] = new_scores[v.vertex_id()] / max_s
            if max(abs(new_scores[v.vertex_id()] - scores[v.vertex_id()]) for v in lc.vertices()) < epsilon:
                return new_scores, it + 1
            scores = new_scores.copy()
        return new_scores, it + 1
    
    def betweenness_centrality(self):
        bc = {}
        delta_init = {0 for i in range(self._graph.order())}
        n = self._graph.order()
        for s in self._graph.vertices:
            delta = delta_init
            (parents, distTo, sigma, stack) = self.bfs(s)
            while stack:
                v = stack.pop()
                for parent in parents[v]:
                    delta[parent] += (sigma[parent] / sigma[v]) * (1 + delta[v])
                if s != v:
                    bc[v] += delta[v]
        for vert in bc.keys():
            bc[vert] *= ((n - 1)*(n - 2)) / 2


        


        
    
    


    
                    

